In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Imports and configuration**

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, accuracy_score,
    classification_report, ConfusionMatrixDisplay
)
from torchvision.transforms import (
    CenterCrop, RandomResizedCrop,
    ColorJitter, Compose, Normalize,
    RandomHorizontalFlip, Resize, ToTensor
)
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import (
    AutoImageProcessor, AutoModelForImageClassification,
    get_cosine_schedule_with_warmup
)

In [5]:
MODEL_ID = "microsoft/swin-tiny-patch4-window7-224"
OUTPUT_DIR = "/content/drive/MyDrive/Fine tuned models/vit_beans_best"
IMAGE_SIZE = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(42)
print("Device: ", device)

Device:  cuda


**Dataset and pretrained model load**

In [7]:
dataset = load_dataset("AI-Lab-Makerere/beans")

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 1034
    })
    validation: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 133
    })
    test: Dataset({
        features: ['image_file_path', 'image', 'labels'],
        num_rows: 128
    })
})


In [9]:
label_names = dataset["train"].features["labels"].names

id2label = {
    class_id: label for class_id, label in enumerate(label_names)
}

label2id = {
    label: class_id for class_id, label in enumerate(label_names)
}

print("Classes: ", label_names)

Classes:  ['angular_leaf_spot', 'bean_rust', 'healthy']


In [10]:
image_processor = AutoImageProcessor.from_pretrained(MODEL_ID)

model = AutoModelForImageClassification.from_pretrained(
    MODEL_ID
).to(device)

model.eval()

preprocessor_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  113MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/221 [00:00<?, ?it/s]

SwinForImageClassification(
  (swin): SwinModel(
    (embeddings): SwinEmbeddings(
      (patch_embeddings): SwinPatchEmbeddings(
        (projection): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      )
      (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): SwinEncoder(
      (layers): ModuleList(
        (0): SwinStage(
          (blocks): ModuleList(
            (0): SwinLayer(
              (attention): SwinAttention(
                (q_proj): Linear(in_features=96, out_features=96, bias=True)
                (k_proj): Linear(in_features=96, out_features=96, bias=True)
                (v_proj): Linear(in_features=96, out_features=96, bias=True)
                (o_proj): Linear(in_features=96, out_features=96, bias=True)
                (relative_position_bias): SwinRelativePositionBias()
              )
              (layernorm_before): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
         